# Final Preprocessing – SMILES to Molecular Graphs

This notebook transforms chemical text into graph structures and prepares three learning stages so a GNN can understand molecules first, interactions second, and finally predict the toxicity of drug combinations.

In [1]:
!pip install rdkit
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

from google.colab import drive
drive.mount('/content/drive')

Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu118.html
  Using cached torch_cluster-1.6.3.tar.gz (54 kB)
  Preparing metadata (setup.py) ... done
  Using cached torch_spline_conv-1.2.2.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done














  Created wheel for torch-cluster: filename=torch_cluster-1.6.3-cp312-cp312-linux_x86_64.whl size=802214 sha256=563cd765d9ba438cf85b5c0175d5506b343d6c4fb0e961d1c81a6d2f985d89cc
  Stored in directory: /root/.cache/pip/wheels/2e/8f/d0/13408a84825c9a587151a74727b4a6d47ec67e0d625b385ad7
  Created wheel for torch-spline-conv: filename=torch_spline_conv-1.2.2-cp312-cp312-linux_x86_64.whl size=300776 sha256=cbeac36ec5fe3b53b7d1274769120ec6d43857c44454d661076fe83066d8cef8
  Stored in directory: /root/.cache/pip/wheels/54/7a/2e/46a729dc0aad2da1a908b0d2ac86ab127d73e6b4310a945d07
Successfully built torch-cluster torch-spline-conv
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive",

## Define Atom and Bond Feature Functions

In [2]:
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

def atom_features(atom):
    """
    Extract a feature vector for an atom.
    Features (11 dimensions):
      - Atom symbol one-hot (C, N, O, S, F, P, Cl, Br, I, B) -> 10 dimensions
      - Degree (one-hot up to 6) -> 7 dimensions
      - Formal charge (continuous, will be scaled later)
      - Hybridization (sp, sp2, sp3) -> 3 dimensions
      - Aromaticity (binary)
      - Atomic mass (continuous)
      - Number of hydrogens (continuous)
    """
    # One-hot for atom symbol (common elements)
    symbol = atom.GetSymbol()
    symbols = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I', 'B']
    symbol_vec = [1.0 if symbol == s else 0.0 for s in symbols]

    # Degree (up to 6)
    degree = atom.GetDegree()
    degree_vec = [0.0]*7
    degree_vec[min(degree, 6)] = 1.0

    # Formal charge
    formal_charge = atom.GetFormalCharge()

    # Hybridization
    hyb = atom.GetHybridization()
    hyb_vec = [
        1.0 if hyb == Chem.rdchem.HybridizationType.SP else 0.0,
        1.0 if hyb == Chem.rdchem.HybridizationType.SP2 else 0.0,
        1.0 if hyb == Chem.rdchem.HybridizationType.SP3 else 0.0
    ]

    # Aromaticity
    is_aromatic = 1.0 if atom.GetIsAromatic() else 0.0

    # Atomic mass
    mass = atom.GetMass()

    # Number of hydrogens (implicit + explicit)
    num_h = atom.GetTotalNumHs()

    features = symbol_vec + degree_vec + [formal_charge] + hyb_vec + [is_aromatic, mass, num_h]
    return np.array(features, dtype=np.float32)

def bond_features(bond):
    """
    Extract features for a bond.
    Features (5 dimensions):
      - Bond type one-hot (single, double, triple, aromatic) -> 4 dimensions
      - Conjugation (binary)
      - In ring (binary)
    """
    bond_type = bond.GetBondType()
    bond_type_vec = [
        1.0 if bond_type == Chem.rdchem.BondType.SINGLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.DOUBLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.TRIPLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.AROMATIC else 0.0
    ]
    is_conjugated = 1.0 if bond.GetIsConjugated() else 0.0
    is_in_ring = 1.0 if bond.IsInRing() else 0.0
    return np.array(bond_type_vec + [is_conjugated, is_in_ring], dtype=np.float32)

These feature definitions follow standard practice in molecular GNNs (e.g., Gilmer et al., 2017; Yang et al., 2019). Including atomic properties like symbol, degree, hybridization, and aromaticity captures essential chemical information, while bond type and conjugation (electrons are shared across multiple adjacent bonds) help the model understand connectivity and electronic effects.

## Convert SMILES to PyG Data Object

In [3]:
from torch_geometric.data import Data

def smiles_to_graph(smiles):
    """
    Convert a SMILES string to a PyTorch Geometric Data object.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Atom features
    atom_feats = []
    for atom in mol.GetAtoms():
        atom_feats.append(atom_features(atom))
    x = torch.tensor(np.array(atom_feats), dtype=torch.float)

    # Bond features (edges)
    edge_indices = []
    edge_feats = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_indices += [[i, j], [j, i]]
        feats = bond_features(bond)
        edge_feats.append(feats)
        edge_feats.append(feats)  # both directions
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(np.array(edge_feats), dtype=torch.float)

    # Create Data object (target will be added later)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    return data

This function transforms a SMILES string into a graph representation that PyTorch Geometric can consume. It creates node features (x), edge indices (connectivity), and edge features (edge_attr). The graph is undirected, so each bond is stored twice (both directions).

## Process Unified Single‑Drug Dataset

 Load the unified CSV, convert each SMILES to a graph, and save a list of Data objects along with the corresponding pLD50 values.

In [4]:
import pandas as pd
import os
import pickle

base_path = '/content/drive/MyDrive/FYP/IRP/Data'
unified_csv = os.path.join(base_path, 'unified_single_drug_pld50.csv')
output_dir = os.path.join(base_path, 'processed_graphs')
os.makedirs(output_dir, exist_ok=True)

# Load unified dataset
df_unified = pd.read_csv(unified_csv)
print(f"Loaded {len(df_unified)} compounds.")

# Convert each SMILES to graph and store
graph_data = []
targets = []
failed = 0
for idx, row in df_unified.iterrows():
    smiles = row['SMILES']
    pld50 = row['pLD50']
    data = smiles_to_graph(smiles)
    if data is None:
        failed += 1
        continue
    data.y = torch.tensor([pld50], dtype=torch.float)
    graph_data.append(data)
    targets.append(pld50)

print(f"Successfully converted {len(graph_data)} molecules. Failed: {failed}")

# Save as a single file using torch.save (list of Data objects)
torch.save(graph_data, os.path.join(output_dir, 'unified_single_drug.pt'))

# Also save targets separately if needed
torch.save(torch.tensor(targets), os.path.join(output_dir, 'unified_targets.pt'))

Loaded 13639 compounds.
Successfully converted 13639 molecules. Failed: 0


Saving the processed graphs avoids recomputing them every time when train. The Data objects contain all graph information; the target y is attached to each object.

## Process MUDI Dataset (Interaction Learning)

In [5]:
mudi_train = os.path.join(base_path, 'MUDI/train_processed.csv')
mudi_test = os.path.join(base_path, 'MUDI/test_processed.csv')

def process_mudi_csv(csv_path, output_name):
    df = pd.read_csv(csv_path)
    graphs_a = []
    graphs_b = []
    labels = []
    failed = 0
    for idx, row in df.iterrows():
        smiles_a = row['smiles_a']
        smiles_b = row['smiles_b']
        label = row['label']
        data_a = smiles_to_graph(smiles_a)
        data_b = smiles_to_graph(smiles_b)
        if data_a is None or data_b is None:
            failed += 1
            continue
        graphs_a.append(data_a)
        graphs_b.append(data_b)
        labels.append(label)
    print(f"{output_name}: processed {len(labels)} pairs, failed {failed}")
    torch.save((graphs_a, graphs_b, torch.tensor(labels)), os.path.join(output_dir, output_name))
    return labels

labels_train = process_mudi_csv(mudi_train, 'mudi_train.pt')
labels_test = process_mudi_csv(mudi_test, 'mudi_test.pt')

mudi_train.pt: processed 221115 pairs, failed 0
mudi_test.pt: processed 89417 pairs, failed 0


For each drug pair, need two graph objects. Store them as two lists in a tuple. The interaction label is stored as a tensor.

In [15]:
df = pd.read_csv(mudi_train)
df['label'].value_counts()

,count
label,
0,186268
1,27320
2,7527


### Handle Class Imbalance

In [6]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights for training set
classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=labels_train)
class_weight_dict = {i: weights[i] for i in range(3)}
print("Class weights:", class_weight_dict)

# Save weights for later use
torch.save(weights, os.path.join(output_dir, 'mudi_class_weights.pt'))

Class weights: {0: np.float64(0.3956933021238216), 1: np.float64(2.697840409956076), 2: np.float64(9.792081838713964)}


So rare class → large weight

common class → small weight

If one interaction type is underrepresented, class weights in the loss function prevent the model from ignoring it. This follows best practices for imbalanced classification (He & Garcia, 2009).

## Process DDI LD₅₀ Smyth Dataset (Fine‑tuning)

In [16]:
smyth_csv = os.path.join(base_path, 'ddi_ld50_smyth1969/ddi_ld50_smyth1969_model_ready.csv')
df_smyth = pd.read_csv(smyth_csv)

graphs_a = []
graphs_b = []
targets = []
failed = 0
for idx, row in df_smyth.iterrows():
    smiles_a = row['SMILES_A']
    smiles_b = row['SMILES_B']
    mix_neglog = row['Mixture_neglog']
    data_a = smiles_to_graph(smiles_a)
    data_b = smiles_to_graph(smiles_b)
    if data_a is None or data_b is None:
        failed += 1
        continue
    graphs_a.append(data_a)
    graphs_b.append(data_b)
    targets.append(mix_neglog)

print(f"Smyth: processed {len(targets)} pairs, failed {failed}")
torch.save((graphs_a, graphs_b, torch.tensor(targets)), os.path.join(output_dir, 'smyth_ddi.pt'))

Smyth: processed 350 pairs, failed 0


The Smyth dataset is small and used for fine‑tuning. Storing it in the same format as MUDI allows reuse of data loaders.

### Summary of Output Files

| File | Contents |
|------|--------|
| `unified_single_drug.pt` | List of PyG Data objects for single drugs (raw pLD₅₀) |
| `unified_targets.pt` | Tensor of raw pLD₅₀ values |
| `pld50_scaler.pkl` | Scaler used for inverse transformation of predictions |
| `mudi_train.pt` | Tuple containing (graphs_a, graphs_b, labels) for training |
| `mudi_test.pt` | Tuple containing (graphs_a, graphs_b, labels) for testing |
| `mudi_class_weights.pt` | Class weight tensor for handling class imbalance |
| `smyth_ddi.pt` | Tuple containing (graphs_a, graphs_b, targets) for regression DDI dataset |
